# Panel macroeconómico con World Bank/WDI

Este ejemplo descarga PIB, población, crecimiento real e inflación para México y Colombia desde World Development Indicators (WDI), y los transforma desde el formato canónico largo a una tabla de comparación.

In [ ]:
from economista import data


## Descargar indicadores WDI

La API acepta varios indicadores en una misma consulta y devuelve un `EconDataset` con su fuente, unidades y metadata reproducible.

In [ ]:
macro = data.fetch(
    source="world_bank",
    dataset="wdi",
    indicator=[
        "NY.GDP.MKTP.CD",      # PIB, USD corrientes
        "SP.POP.TOTL",         # población total
        "NY.GDP.MKTP.KD.ZG",   # crecimiento real del PIB, % anual
        "FP.CPI.TOTL.ZG",      # inflación CPI, % anual
    ],
    geo=["MEX", "COL"],
    start=2010,
    end=2024,
)

frame = macro.to_pandas()
frame[["geo", "time", "indicator", "indicator_name", "value", "unit"]].head()

## Convertir a una tabla de comparación

El formato largo es el intercambio canónico de `economista`. Para comparar indicadores por país y año, pivotamos temporalmente y expresamos los niveles en billions y millions.

In [ ]:
macro_panel = (
    frame.pivot(index=["time", "geo"], columns="indicator", values="value")
    .rename(
        columns={
            "NY.GDP.MKTP.CD": "gdp_usd",
            "SP.POP.TOTL": "population",
            "NY.GDP.MKTP.KD.ZG": "real_gdp_growth_percent",
            "FP.CPI.TOTL.ZG": "inflation_percent",
        }
    )
    .assign(
        gdp_usd_billions=lambda data: data["gdp_usd"] / 1_000_000_000,
        population_millions=lambda data: data["population"] / 1_000_000,
    )
    .sort_index()
)

macro_panel[
    [
        "gdp_usd_billions",
        "population_millions",
        "real_gdp_growth_percent",
        "inflation_percent",
    ]
].tail(6).round(1)

## Revisar metadata

Al trabajar con varios indicadores, la tabla conserva el nombre y unidad por observación. La metadata del `EconDataset` registra además la fuente, cobertura, fecha de descarga y consulta original.

In [ ]:
macro.metadata.to_dict()